# 03 Data Cleaning Personal Finance

Notebook ini fokus hanya pada dataset `data/raw/personal_finance/Personal_Finance_Dataset.csv`.

Tujuan:
- Baca dan bedah data personal finance.
- Bersihkan tanggal, kategori, amount, tipe transaksi, dan duplikasi.
- Buat dataset rapi untuk analisis personal finance.
- Simpan 1 file clean ke `data/clean/personal_finance_clean.csv`.

Output utama:
- `data_bersih`: dataset final di memory.
- `personal_finance_clean.csv`: satu file clean final, amount langsung memakai IDR.


## 1. Import library

Pakai library ringan: `pandas`, `numpy`, dan `matplotlib`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')


## 2. Konfigurasi path

Raw data dibaca dari folder `data/raw/personal_finance`. Hasil akhir disimpan sebagai satu dataset di `data/clean`.


In [ ]:
root_project = Path.cwd().resolve()
if root_project.name == 'notebooks':
    root_project = root_project.parent

file_raw = root_project / 'data' / 'raw' / 'personal_finance' / 'Personal_Finance_Dataset.csv'
folder_clean = root_project / 'data' / 'clean'
folder_clean.mkdir(parents=True, exist_ok=True)

print('Root project :', root_project)
print('File raw     :', file_raw)
print('Folder clean :', folder_clean)

if not file_raw.exists():
    raise FileNotFoundError(f'File raw tidak ditemukan: {file_raw}')


## 3. Load data raw

Dataset punya 5 kolom utama: tanggal, deskripsi, kategori, amount, dan tipe transaksi.


In [ ]:
data_mentah = pd.read_csv(file_raw)

print('Shape data mentah:', data_mentah.shape)
display(data_mentah.sample(20))
display(pd.DataFrame({'kolom': data_mentah.columns, 'tipe_data': data_mentah.dtypes.astype(str).values}))


## 4. Bedah kualitas data awal

Cek awal:
- missing value
- duplikasi
- jumlah income/expense
- kategori
- statistik amount
- outlier IQR


In [ ]:
missing_awal = data_mentah.isna().sum().reset_index(name='jumlah_missing').rename(columns={'index': 'kolom'})
missing_awal['persen_missing'] = (missing_awal['jumlah_missing'] / len(data_mentah) * 100).round(2)

duplikat_exact = int(data_mentah.duplicated().sum())
duplikat_key = int(data_mentah.duplicated(subset=['Date', 'Transaction Description', 'Category', 'Amount', 'Type'], keep=False).sum())

amount_awal = pd.to_numeric(data_mentah['Amount'], errors='coerce')
q1_awal = amount_awal.quantile(0.25)
q3_awal = amount_awal.quantile(0.75)
iqr_awal = q3_awal - q1_awal
batas_bawah_awal = q1_awal - 1.5 * iqr_awal
batas_atas_awal = q3_awal + 1.5 * iqr_awal
outlier_awal = int(((amount_awal < batas_bawah_awal) | (amount_awal > batas_atas_awal)).sum())

print('Baris raw:', len(data_mentah))
print('Duplikat exact:', duplikat_exact)
print('Duplikat key:', duplikat_key)
print('Amount <= 0:', int((amount_awal <= 0).sum()))
print('Outlier IQR:', outlier_awal)
print('Batas atas IQR:', round(batas_atas_awal, 2))

display(missing_awal)
display(data_mentah['Type'].value_counts(dropna=False).reset_index(name='jumlah_baris'))
display(data_mentah['Category'].value_counts(dropna=False).reset_index(name='jumlah_baris'))
display(data_mentah['Amount'].describe().reset_index().rename(columns={'index': 'metric', 'Amount': 'value'}))


## 6. Helper cleaning

Logic helper:
- Bersihkan teks kosong.
- Parse tanggal.
- Standarkan kategori ke kategori Indonesia.


In [ ]:
def bersihkan_teks(value, default='unknown'):
    if pd.isna(value):
        return default
    value = str(value).strip()
    return value if value else default


def ubah_tanggal(series):
    return pd.to_datetime(series, errors='coerce', format='mixed')


def kategori_standar(kategori):
    teks = str(kategori).lower().strip()

    if teks == 'food & drink':
        return 'Makanan & Minuman'
    if teks == 'shopping':
        return 'Belanja'
    if teks == 'travel':
        return 'Transportasi & Travel'
    if teks == 'health & fitness':
        return 'Kesehatan'
    if teks in ['rent', 'utilities']:
        return 'Tagihan & Rumah'
    if teks == 'entertainment':
        return 'Hiburan'
    if teks in ['salary', 'investment']:
        return 'Pendapatan & Investasi'
    return 'Lainnya'


## 7. Cleaning utama

Logic cleaning:
1. Rename kolom agar rapi dan mudah dibaca.
2. Parse `Date` menjadi `tanggal_transaksi`.
3. Bersihkan tipe transaksi.
4. Ubah `Amount` langsung menjadi `amount_idr`.
5. Buat `kategori_clean` langsung dari kategori raw.
6. Hapus duplikasi jika ada.
7. Ambil fitur final yang dipakai saja.


In [ ]:
data_bersih = data_mentah.copy()

data_bersih = data_bersih.rename(columns={
    'Date': 'tanggal_transaksi',
    'Category': 'kategori_clean',
    'Amount': 'amount_idr',
    'Type': 'tipe_transaksi',
})

data_bersih['tanggal_transaksi'] = ubah_tanggal(data_bersih['tanggal_transaksi'])
data_bersih['kategori_clean'] = data_bersih['kategori_clean'].map(kategori_standar)
data_bersih['tipe_transaksi'] = data_bersih['tipe_transaksi'].map(bersihkan_teks)
data_bersih['amount_idr'] = pd.to_numeric(data_bersih['amount_idr'], errors='coerce').astype(float)

data_bersih = data_bersih.dropna(subset=['tanggal_transaksi', 'amount_idr', 'kategori_clean', 'tipe_transaksi']).copy()
data_bersih = data_bersih[data_bersih['amount_idr'] > 0].copy()

baris_sebelum_duplikat = len(data_bersih)
data_bersih = data_bersih.drop_duplicates(subset=['tanggal_transaksi', 'kategori_clean', 'amount_idr', 'tipe_transaksi']).copy()
duplikat_dihapus = baris_sebelum_duplikat - len(data_bersih)

data_bersih['tahun'] = data_bersih['tanggal_transaksi'].dt.year
data_bersih['bulan'] = data_bersih['tanggal_transaksi'].dt.month
data_bersih['tahun_bulan'] = data_bersih['tanggal_transaksi'].dt.to_period('M').astype(str)

data_bersih = data_bersih.sort_values('tanggal_transaksi').reset_index(drop=True)

display(data_bersih.head())
print('Baris raw:', len(data_mentah))
print('Baris clean:', len(data_bersih))
print('Duplikat dihapus:', duplikat_dihapus)


## 8. Validasi hasil cleaning

Validasi singkat:
- Kolom penting tidak missing.
- Amount positif.
- Tipe transaksi valid: `Income` atau `Expense`.
- Tidak ada duplikasi key.


In [ ]:
kolom_penting = ['tanggal_transaksi', 'kategori_clean', 'amount_idr', 'tipe_transaksi']

validasi_bersih = pd.DataFrame({
    'kolom': kolom_penting,
    'jumlah_missing': [int(data_bersih[kolom].isna().sum()) for kolom in kolom_penting],
})
validasi_bersih['persen_missing'] = (validasi_bersih['jumlah_missing'] / len(data_bersih) * 100).round(2)

masalah_validasi = {
    'amount_tidak_valid': int((data_bersih['amount_idr'] <= 0).sum()),
    'tipe_tidak_valid': int((~data_bersih['tipe_transaksi'].isin(['Income', 'Expense'])).sum()),
    'duplikat_key': int(data_bersih.duplicated(subset=['tanggal_transaksi', 'kategori_clean', 'amount_idr', 'tipe_transaksi']).sum()),
}

display(validasi_bersih)
print(masalah_validasi)

if validasi_bersih['jumlah_missing'].sum() > 0 or sum(masalah_validasi.values()) > 0:
    raise ValueError('Masih ada masalah pada data bersih.')

print('Validasi cleaning lolos.')


## 9. Analisis singkat setelah cleaning

Cek hasil final: ringkasan per tahun, kategori, dan tipe transaksi.


In [ ]:
ringkasan_bersih = (
    data_bersih.groupby(['tahun', 'tipe_transaksi'])
    .agg(
        jumlah_transaksi=('amount_idr', 'count'),
        total_idr=('amount_idr', 'sum'),
        median_amount_idr=('amount_idr', 'median'),
    )
    .reset_index()
)

ringkasan_kategori = (
    data_bersih.groupby(['kategori_clean', 'tipe_transaksi'])
    .agg(jumlah_transaksi=('amount_idr', 'count'), total_idr=('amount_idr', 'sum'))
    .reset_index()
    .sort_values('jumlah_transaksi', ascending=False)
)

display(ringkasan_bersih)
display(ringkasan_kategori)


## 10. Dataset final

`data_bersih` adalah satu dataset utama. Income dan expense tidak dipisah file; cukup filter kolom `tipe_transaksi` saat analisis.


In [ ]:
kolom_final = [
    'tanggal_transaksi', 'tahun', 'bulan', 'tahun_bulan',
    'tipe_transaksi', 'kategori_clean', 'amount_idr',
]

data_bersih = data_bersih[kolom_final].copy()

display(data_bersih.head())
print('Total clean rows:', len(data_bersih))
print('Income rows:', int((data_bersih['tipe_transaksi'] == 'Income').sum()))
print('Expense rows:', int((data_bersih['tipe_transaksi'] == 'Expense').sum()))


## 11. Simpan 1 dataset clean

Hanya satu file disimpan: `personal_finance_clean.csv`.

Kolom final yang disimpan hanya fitur dipakai: tanggal, tahun, bulan, tahun_bulan, tipe_transaksi, kategori_clean, dan amount_idr.


In [ ]:
file_output = folder_clean / 'personal_finance_clean.csv'

data_bersih.to_csv(file_output, index=False)

print('File clean tersimpan:', file_output)
print('Jumlah baris:', len(data_bersih))
print('Jumlah kolom:', data_bersih.shape[1])
